# RUN ALL — NB1 → NB7 한 번에
각 NB 파일(NB1~NB7)의 셀을 **그대로 읽어서 순서대로 실행**합니다. NB 파일은 수정하지 않습니다.
- 건너뛰는 셀: 각 NB의 셀 0(설정, 여기서 대신 함), `시각화` 셀, NB1 미리보기 영상, NB2 `5-B (재시작 시에만)`, NB4 오버레이 미리보기
- 각 NB가 끝나면 결과 파일이 **이번 실행에서 새로 생겼는지** 확인하고, 없으면 멈춥니다.
- 끊기면 `START_AT` 을 멈춘 NB 번호로 바꿔 다시 실행 (NB2 는 끝난 프레임을 건너뜀).
- 런타임: **GPU (T4 이상)**. 브라우저 탭은 열어 두세요.

In [ ]:
#@title 설정
ROOT       = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION    = "0729_walk_3kmh_5500"                   #@param {type:"string"}
R3D_PATH   = "/content/drive/MyDrive/walking_0729/raw/260729_60fps_walkingonly.r3d"  #@param {type:"string"}
START_FID  = 5500   #@param {type:"integer"}
N_FRAMES   = 250    #@param {type:"integer"}
STEP       = 2      #@param {type:"integer"}
IMU_PREFIX = "MT_01200295_002_60fps_walking"  #@param ["MT_01200295_001_full", "MT_01200295_002_60fps_walking"]
JUMP_FID   = 0      #@param {type:"integer"}
START_AT   = 5      #@param {type:"integer"}
END_AT     = 7      #@param {type:"integer"}
LOCAL_CACHE = True  #@param {type:"boolean"}
PERSONAL_MESH = ""  #@param {type:"string"}
QUICK_FRAMES = 84    #@param {type:"integer"}
CLOTH_ONESIDED = True  #@param {type:"boolean"}
# True: 반바지(하의) 정점은 "몸이 옷 밖으로 나오면 벌점"만 (BUFF/DoubleFusion 방식). 결과 폴더에 _c1
# 빠른 확인: NB5~7 을 앞쪽 N 프레임만 (예: 84 = 보행 2주기). 0 이면 전부. 결과 폴더에 _q{N} 이 붙음
# 개인 메쉬 npz 경로 (예: f"{ROOT}/assets/personal_mesh.npz"). 비우면 기존 β. 지정하면 결과는 04_fit_personal 등 별도 폴더에 저장
# NB5~ 는 depth/video 를 여러 번 읽기만 하므로 Colab 로컬 디스크로 한 번 복사해 씀 (Drive 읽기 병목 제거)

OVERRIDES = dict(R3D_PATH=R3D_PATH, START_FID=START_FID, N_FRAMES=N_FRAMES, STEP=STEP,
                 IMU_PREFIX=IMU_PREFIX, JUMP_FID=JUMP_FID)

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
os.environ.pop("WALK_DEPTH_SEQ", None); os.environ.pop("WALK_VIDEO_SEQ", None)   # 이전 세션의 로컬 복사본 경로 초기화
os.environ.pop("WALK_PERSONAL", None); os.environ.pop("WALK_FIT_TAG", None)
os.environ.pop("WALK_FRAME_LIMIT", None)
os.environ.pop("WALK_CLOTH_ONESIDED", None)
if CLOTH_ONESIDED:
    os.environ["WALK_CLOTH_ONESIDED"] = "1"; print("하의 한쪽 데이터 항 사용")
if QUICK_FRAMES > 0:
    os.environ["WALK_FRAME_LIMIT"] = str(QUICK_FRAMES); print(f"빠른 확인: 앞쪽 {QUICK_FRAMES} 프레임만")
if PERSONAL_MESH:
    assert os.path.exists(PERSONAL_MESH), PERSONAL_MESH
    os.environ["WALK_PERSONAL"] = PERSONAL_MESH
    print("개인 메쉬 사용:", PERSONAL_MESH, "→ 결과 폴더 접미사 _personal")
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
print("세션 폴더:", walkcfg.SESS)
import torch; assert torch.cuda.is_available(), "런타임 → 런타임 유형 변경 → GPU"


In [ ]:
#@title 실행기
import json, re, time, glob, gc, os
from IPython import get_ipython
ip = get_ipython()

SKIP = [  # (NB, 셀 내용에 들어있는 문자열) → 건너뜀
    (0, "#@title 시각화"), (0, "#@title 0. 설정"),
    (1, "preview_fid.mp4"), (2, "(재시작 시에만)"), (4, "overlay_subset"),
]
def nb_path(k): return glob.glob(f"{ROOT}/notebooks/NB{k}_*.ipynb")[0]

def patch(src):
    for name, val in OVERRIDES.items():
        src = re.sub(rf"^(\s*){name}\s*=.*$", lambda m: f"{m.group(1)}{name} = {val!r}", src, flags=re.M)
    return src

def fresh(paths, t0, n_min=1):
    ok = [p for p in paths if os.path.exists(p) and os.path.getmtime(p) >= t0 - 5]
    return len(ok) >= n_min

def check(k, t0):
    importlib.reload(walkcfg)
    E = walkcfg.EXTRACT_DIR
    if k == 1:
        n = len(glob.glob(f"{E}/rgb/*.jpg")); return n > 0, f"rgb {n}장"
    if k == 2:
        n_rgb = len(glob.glob(f"{E}/rgb/*.jpg"))   # 기준 = 실제 추출된 프레임 수 (프레임 목록 추출이면 N_FRAMES 와 다름)
        p = glob.glob(f"{walkcfg.DEPTH_SEQ}/WALK_side_*_person_metric.npy"); return len(p) >= n_rgb*0.95, f"depth {len(p)}/{n_rgb}"
    if k == 3:
        ok = fresh([walkcfg.GLOBAL_JSON], t0)
        a = json.load(open(walkcfg.GLOBAL_JSON))["chosen_global_a"] if ok else None
        if a is not None and not (0.40 <= a <= 0.55):
            print(f"⚠ a = {a:.3f} — 001/4-20 범위(0.45~0.50) 밖. 계속 진행하지만 결과 보고 때 확인 필요")
        return ok, f"a = {a}"
    if k == 4: return fresh([walkcfg.FOURDH_SEQ], t0), "4D-Humans track"
    if k == 5: return fresh([walkcfg.NEAR_F3, walkcfg.FAR_F2], t0, 2), "near f3 + far2"
    if k == 6: return fresh([f"{walkcfg.KNEE_DIR}/far_knee_final.npz"], t0), "far_knee_final"
    if k == 7: return fresh([f"{walkcfg.IMU_OUT}/imu_validation.json"], t0), "imu_validation"

# NB4: 4D-Humans 공식 체크포인트는 omegaconf 객체를 담고 있어 PyTorch 2.6+ 의 weights_only 기본값으로는 못 읽음.
#      원래 NB4 에서는 건너뛴 오버레이 셀이 이 환경변수를 설정했으므로 여기서 대신 설정 (공식 배포 파일만 읽음).
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
T_ALL = time.time()
for __nbk in range(START_AT, END_AT + 1):   # 고유 이름: 노트북 셀이 k, t0 등을 덮어써도 영향 없게
    os.chdir("/content")                      # NB2 가 Drive 로 chdir 하므로 매 NB 마다 원위치
    gc.collect(); torch.cuda.empty_cache()
    if __nbk >= 5 and LOCAL_CACHE and not os.environ.get("WALK_DEPTH_SEQ"):
        import shutil
        src_d, src_v = walkcfg.DEPTH_SEQ, walkcfg.VIDEO_SEQ
        dst = f"/content/walk_cache/{SESSION}"
        print(f"[cache] depth/video → {dst} 로 복사 (한 번만)", flush=True); t_c = time.time()
        os.makedirs(f"{dst}/out", exist_ok=True); os.makedirs(f"{dst}/video", exist_ok=True)
        def _copy(src, g):
            """임시 이름으로 복사 → 완료 후 이름 변경. 크기가 같으면 건너뜀. Drive 끊기면 재연결 후 재시도."""
            if os.path.exists(g) and os.path.getsize(g) == os.path.getsize(src):
                return
            for attempt in range(3):
                try:
                    shutil.copy(src, g + ".part"); os.replace(g + ".part", g); return
                except OSError as e:
                    print(f"  [cache] {os.path.basename(src)} 복사 실패 ({e}) → Drive 재연결 후 재시도 {attempt + 1}/3", flush=True)
                    from google.colab import drive as _d
                    try: _d.flush_and_unmount()
                    except Exception: pass
                    _d.mount("/content/drive", force_remount=True); time.sleep(5)
            raise RuntimeError(f"복사 실패: {src}")
        dfiles = sorted(glob.glob(f"{src_d}/*_person_metric.npy"))
        if QUICK_FRAMES > 0:
            dfiles = dfiles[:QUICK_FRAMES]                      # 빠른 확인이면 필요한 프레임만
        keys = [os.path.basename(f).replace("_person_metric.npy", "") for f in dfiles]
        for i_c, f in enumerate(dfiles):
            _copy(f, f"{dst}/out/{os.path.basename(f)}")
            v = f"{src_v}/{keys[i_c]}.npz"
            if os.path.exists(v):
                _copy(v, f"{dst}/video/{keys[i_c]}.npz")
            if i_c % 25 == 0:
                print(f"  [cache] {i_c + 1}/{len(dfiles)}", flush=True)
        os.environ["WALK_DEPTH_SEQ"] = f"{dst}/out"; os.environ["WALK_VIDEO_SEQ"] = f"{dst}/video"
        importlib.reload(walkcfg)
        print(f"[cache] 완료 {(time.time()-t_c)/60:.1f}분 — 이후 스크립트는 로컬 복사본을 읽음", flush=True)
    nb = json.load(open(nb_path(__nbk), encoding="utf-8"))
    __nbt0 = time.time()
    print(f"\n{'#'*70}\n# NB{__nbk}  {os.path.basename(nb_path(__nbk))}\n{'#'*70}", flush=True)
    for i, c in enumerate(nb["cells"]):
        if c["cell_type"] != "code": continue
        src = "".join(c["source"])
        if not src.strip(): continue
        if any((s_nb in (0, __nbk)) and s in src for s_nb, s in SKIP): continue
        title = next((l for l in src.splitlines() if l.strip()), "")[:70]
        print(f"\n--- NB{__nbk} 셀 {i}: {title}", flush=True)
        r = ip.run_cell(patch(src))
        if not r.success:
            raise RuntimeError(f"NB{__nbk} 셀 {i} 에서 오류 → 고친 뒤 START_AT={__nbk} 로 다시 실행")
    if __nbk == 5:   # far 다리 튐 프레임: 앞뒤 정상 프레임 보간을 초기값으로 다시 피팅 (안 되면 NB6 에서 보간)
        print("--- NB5+ far 튐 프레임 재피팅 (far_spike_refit.py)", flush=True)
        r = ip.run_cell(f'!cd "{ROOT}/scripts" && python far_spike_refit.py 2>&1 | grep -v "side v2"')
    ok, msg = check(__nbk, __nbt0)
    print(f"\n[NB{__nbk}] {'OK' if ok else '실패'} — {msg}  ({(time.time()-__nbt0)/60:.1f}분)", flush=True)
    if not ok:
        raise RuntimeError(f"NB{__nbk} 결과 파일이 이번 실행에서 안 만들어짐 → 위 로그 확인 후 START_AT={__nbk}")
    if __nbk == 2:   # Sapiens 모델 GPU 메모리 해제 (NB4 전에)
        for v in ("seg_model", "depth_model", "normal_model"):
            globals().pop(v, None)
        gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ NB{START_AT}~NB{END_AT} 완료  총 {(time.time()-T_ALL)/60:.1f}분")


In [ ]:
#@title 결과 요약 (있는 것만)
import json, os, importlib, numpy as np
from IPython.display import Image, display
importlib.reload(walkcfg)
if os.path.exists(walkcfg.GLOBAL_JSON):
    g = json.load(open(walkcfg.GLOBAL_JSON)); print("a =", round(g["chosen_global_a"], 3))
if os.path.exists(walkcfg.FAR_F2):
    lab = np.load(walkcfg.FAR_F2, allow_pickle=True)["label"].astype(str)
    print("far:", dict(zip(*np.unique(lab, return_counts=True))))
_v = f"{walkcfg.IMU_OUT}/imu_validation.json"
if os.path.exists(_v):
    print(json.dumps(json.load(open(_v)), indent=1, ensure_ascii=False)[:1500])
else:
    print("(NB7 결과 아직 없음)")
for p in (f"{walkcfg.KNEE_DIR}/far_knee_final_v2.png", f"{walkcfg.IMU_OUT}/imu_validation.png"):
    if os.path.exists(p): display(Image(p, width=1000))
